In [ ]:
import json
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import (
    average_precision_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


In [ ]:
TRAIN_FEAT_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/train_features.csv"
TRAIN_TARGET_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/customer_clv_train.csv"
BEST_PARAMS_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/best_params.json"


In [ ]:
# Load train data and build modeling table
required_paths = [TRAIN_FEAT_PATH, TRAIN_TARGET_PATH]
missing_required = [p for p in required_paths if not os.path.exists(p)]
if missing_required:
    raise FileNotFoundError(f"Missing required files: {missing_required}")

train_features = pd.read_csv(TRAIN_FEAT_PATH)
train_target = pd.read_csv(TRAIN_TARGET_PATH)

train_df = train_target.merge(train_features, on="cust_id", how="inner", validate="one_to_one")
feature_cols = [c for c in train_df.columns if c not in ["cust_id", "revenue_2018_2019"]]
X = train_df[feature_cols].copy()
y = train_df["revenue_2018_2019"].to_numpy()
y_bin = (y > 0).astype(int)

print("train_df shape:", train_df.shape)
print("X shape:", X.shape)
print("positive ratio:", y_bin.mean())


In [ ]:
# Load tuned params if available; fallback to defaults
clf_params = {
    "iterations": 300,
    "depth": 6,
    "learning_rate": 0.05,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_state": 42,
    "verbose": False,
}

reg_params = {
    "iterations": 400,
    "depth": 6,
    "learning_rate": 0.05,
    "loss_function": "MAE",
    "random_state": 42,
    "verbose": False,
}

if os.path.exists(BEST_PARAMS_PATH):
    with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
        best_params = json.load(f)
    clf_params.update(best_params.get("classifier", {}).get("best_params", {}))
    reg_params.update(best_params.get("regressor", {}).get("best_params", {}))
    print(f"Loaded tuned params from: {BEST_PARAMS_PATH}")
else:
    print("best_params.json not found. Using defaults.")

print("Classifier params:", clf_params)
print("Regressor params:", reg_params)


In [ ]:
# Holdout validation
X_tr, X_val, y_tr, y_val, yb_tr, yb_val = train_test_split(
    X,
    y,
    y_bin,
    test_size=0.2,
    random_state=42,
    stratify=y_bin,
)

# Stage 1: classifier
clf = CatBoostClassifier(**clf_params)
clf.fit(X_tr, yb_tr)
tr_prob = clf.predict_proba(X_tr)[:, 1]
val_prob = clf.predict_proba(X_val)[:, 1]

# Stage 2: regressor with classifier output as feature
X_tr_reg = X_tr.copy()
X_val_reg = X_val.copy()
X_tr_reg["prob_return"] = tr_prob
X_val_reg["prob_return"] = val_prob

reg = CatBoostRegressor(**reg_params)
reg.fit(X_tr_reg, np.log1p(y_tr))

val_log = reg.predict(X_val_reg)
y_pred = np.clip(np.expm1(val_log), 0, None)

# Baseline for comparison (predict train mean)
baseline_pred = np.full_like(y_val, fill_value=y_tr.mean(), dtype=float)

# Classification metrics
auc = roc_auc_score(yb_val, val_prob)
ap = average_precision_score(yb_val, val_prob)

# Regression metrics (on all customers)
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

# Regression metrics on positive spenders only
pos_mask = y_val > 0
if pos_mask.sum() > 0:
    mae_pos = mean_absolute_error(y_val[pos_mask], y_pred[pos_mask])
    rmse_pos = np.sqrt(mean_squared_error(y_val[pos_mask], y_pred[pos_mask]))
else:
    mae_pos = np.nan
    rmse_pos = np.nan

baseline_mae = mean_absolute_error(y_val, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_pred))


In [ ]:
metrics = pd.DataFrame(
    [
        ["classifier_auc", auc],
        ["classifier_avg_precision", ap],
        ["reg_mae_all", mae],
        ["reg_rmse_all", rmse],
        ["reg_r2_all", r2],
        ["reg_mae_positive_only", mae_pos],
        ["reg_rmse_positive_only", rmse_pos],
        ["baseline_mae_all", baseline_mae],
        ["baseline_rmse_all", baseline_rmse],
    ],
    columns=["metric", "value"],
)

print("Validation metrics (holdout):")
print(metrics.to_string(index=False))

improvement_mae = baseline_mae - mae
improvement_rmse = baseline_rmse - rmse
print(f"\nImprovement vs baseline (MAE): {improvement_mae:.6f}")
print(f"Improvement vs baseline (RMSE): {improvement_rmse:.6f}")


In [ ]:
# Optional: save metrics
METRICS_OUT_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/validation_metrics.csv"
metrics.to_csv(METRICS_OUT_PATH, index=False)
print(f"Saved metrics to: {METRICS_OUT_PATH}")
